# Запуск и проверка Predictive Maintenance MLOps System (отчет)

Этот ноутбук фиксирует воспроизводимый запуск MLOps-системы для задачи предиктивного обслуживания промышленного оборудования.

Цель отчёта — показать, что проект можно поднять из репозитория, проверить тестами, запустить инфраструктуру, открыть основные UI, выполнить inference-запросы, проверить мониторинг, SLO rules, data drift report и canary traffic switching.

Проект включает:

- FastAPI inference service;
- Feast Feature Store;
- Redis Online Store;
- PostgreSQL;
- MLflow Tracking Server и Model Registry;
- Airflow orchestration;
- Prometheus monitoring;
- Grafana dashboard;
- Node Exporter infrastructure monitoring;
- Evidently data drift report;
- Nginx-based canary gateway;
- Docker Compose и Ansible как Infrastructure as Code.

Все команды ниже предполагают запуск из корня репозитория `predictive-maintenance-mlops`.

# Порты 

```text
8000   FastAPI API
8010   Canary Gateway
5050   MLflow UI
8081   Airflow UI
9090   Prometheus
9100   Node Exporter
3000   Grafana
15432  PostgreSQL
16379  Redis
```

# Корень проекта

In [9]:
%env PROJECT_ROOT=/Users/perceivery/Desktop/predictive-maintenance-mlops

env: PROJECT_ROOT=/Users/perceivery/Desktop/predictive-maintenance-mlops


# Проверка файлов проекта

In [10]:
%%bash
cd "$PROJECT_ROOT"
pwd
ls -la | head

/Users/perceivery/Desktop/predictive-maintenance-mlops
total 104
drwxr-xr-x@ 29 perceivery  staff    928 May 30 20:46 .
drwx------@ 59 perceivery  staff   1888 May 30 20:46 ..
-rw-r--r--@  1 perceivery  staff    496 May 25 17:25 .env.example
drwxr-xr-x@ 13 perceivery  staff    416 May 30 20:38 .git
drwxr-xr-x@  3 perceivery  staff     96 May 27 14:45 .github
-rw-r--r--@  1 perceivery  staff    319 May 27 18:31 .gitignore
drwxr-xr-x@  6 perceivery  staff    192 May 27 14:45 .pytest_cache
drwxr-xr-x@  7 perceivery  staff    224 May 30 20:06 .venv
-rw-r--r--@  1 perceivery  staff   2466 May 27 17:58 Makefile


# Проверка состояния репозитория

Перед запуском сервисов фиксируем состояние Git-репозитория.

In [11]:
%%bash
cd "$PROJECT_ROOT"
git status
git log --oneline -8

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/service_up_report.ipynb

nothing added to commit but untracked files present (use "git add" to track)
2cb17ed Add MDD latency A/B test notebook and ADR reference
fdcd059 Add canary and infrastructure health checks to Ansible vars
60893b7 Extend Ansible deployment with canary and smoke checks
09fb502 Ignore local MLflow file store
722e719 Add Makefile commands for canary deployment
e3651eb Show pytest summary while suppressing warnings
773cc70 Disable third-party warnings output in pytest
842fda1 Document canary deployment for inference service


# Проверка локального окружения

Проверяем, что доступны Python, Docker, Docker Compose и Makefile. Эти инструменты нужны для запуска тестов, сборки контейнеров и поднятия MLOps-инфраструктуры.

In [12]:
%%bash
cd "$PROJECT_ROOT"
python --version
docker --version
docker compose version
make --version | head -n 1

Python 3.12.0
Docker version 29.1.2, build 890dcca
Docker Compose version v2.40.3-desktop.1
GNU Make 3.81


# Создание venv  и установка зависимостей

Создаём локальное Python-окружение `.venv` и устанавливаем зависимости проекта из `requirements.txt`.

В проекте установка обёрнута в Makefile-команду

```bash
make install
```

In [ ]:
%%bash
cd "$PROJECT_ROOT"
if [ ! -d ".venv" ]; then
  python3 -m venv .venv
fi

. .venv/bin/activate

In [ ]:
%%bash
cd "$PROJECT_ROOT"
make install

# Запуск тестов проекта

После проверки окружения запускаем автоматические тесты проекта.

Команда `make test` использует pytest и проверяет базовую работоспособность API-логики и вспомогательных компонентов.

Ожидаемый результат:

```text
9 passed
```

In [13]:
%%bash
cd "$PROJECT_ROOT"
make test

python -m pytest -q
.........                                                                [100%]
9 passed in 1.86s


# Проверка Docker Compose конфигурации основного контура

Перед запуском сервисов проверяем, что основной Docker Compose файл синтаксически корректен и может быть собран Docker Compose.

Основной контур описан в файле:

```text
infra/docker-compose.yml
```
В него входят PostgreSQL, Redis, MLflow, FastAPI, Airflow, Prometheus, Grafana и Node Exporter.

In [14]:
%%bash
cd "$PROJECT_ROOT"
docker compose -f infra/docker-compose.yml config >/tmp/main-compose-config.txt
echo "Main Docker Compose config is valid."

Main Docker Compose config is valid.


# Проверка Docker Compose конфигурации canary-контура

Canary-контур описан в отдельном файле:

```text
infra/docker-compose.canary.yml
```

Он поднимает два экземпляра inference service:

- stable;
- canary;

и Nginx gateway, который переключает трафик между ними.

In [15]:
%%bash
cd "$PROJECT_ROOT"
docker compose -f infra/docker-compose.canary.yml config >/tmp/canary-compose-config.txt
echo "Canary Docker Compose config is valid."

Canary Docker Compose config is valid.


# Запуск основного MLOps-контура

Поднимаем основной Docker Compose контур из файла:

```text
infra/docker-compose.yml
```

В этом контуре запускаются основные сервисы проекта: PostgreSQL, Redis, MLflow, FastAPI, Airflow, Prometheus, Grafana и Node Exporter.

Команда выполняет сборку образов при необходимости и запускает контейнеры в фоне.

In [17]:
%%bash
cd "$PROJECT_ROOT"
docker compose -f infra/docker-compose.yml up -d --build

#1 [internal] load local bake definitions
#1 reading from stdin 2.12kB done
#1 DONE 0.0s

#2 [api internal] load build definition from Dockerfile.api
#2 transferring dockerfile: 680B done
#2 DONE 0.0s

#3 [airflow-scheduler internal] load build definition from Dockerfile.airflow
#3 transferring dockerfile: 231B done
#3 DONE 0.0s

#4 [airflow-webserver internal] load metadata for docker.io/apache/airflow:2.10.4-python3.12
#4 DONE 4.2s

#5 [airflow-init internal] load .dockerignore
#5 transferring context: 2B done
#5 DONE 0.0s

#6 [airflow-scheduler 1/3] FROM docker.io/apache/airflow:2.10.4-python3.12@sha256:04e7fba174eb5c77057f59ef18c1215179181cd5dd7189afbc39b1146f54540f
#6 resolve docker.io/apache/airflow:2.10.4-python3.12@sha256:04e7fba174eb5c77057f59ef18c1215179181cd5dd7189afbc39b1146f54540f done
#6 DONE 0.0s

#7 [airflow-scheduler internal] load build context
#7 transferring context: 336B done
#7 DONE 0.0s

#8 [airflow-scheduler 2/3] COPY infra/requirements-airflow.txt /tmp/requirem

 infra-airflow-webserver  Built
 infra-airflow-init  Built
 infra-airflow-scheduler  Built
 infra-api  Built
time="2026-05-30T20:54:13+03:00" level=warning msg="Found orphan containers ([predictive-maintenance-canary-gateway predictive-maintenance-api-canary predictive-maintenance-api-stable molvit-minio]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."
 Container predictive-maintenance-airflow-init  Recreate
 Container predictive-maintenance-api  Recreate
 Container predictive-maintenance-api  Recreated
 Container predictive-maintenance-airflow-init  Recreated
 Container predictive-maintenance-airflow-webserver  Recreate
 Container predictive-maintenance-airflow-scheduler  Recreate
 Container predictive-maintenance-airflow-webserver  Recreated
 Container predictive-maintenance-airflow-scheduler  Recreated
 Container predictive-maintenance-redis  Starting
 Container predictive-mainte

# Проверка статусов контейнеров основного контура

После запуска проверяем состояние контейнеров. Для backend-сервисов важно увидеть, что контейнеры находятся в статусе `Up`, а сервисы с healthcheck имеют статус `healthy`.

Эта проверка нужна для подтверждения, что заявленные компоненты MLOps-системы действительно запущены.

In [18]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.yml ps

NAME                                       IMAGE                           COMMAND                  SERVICE             CREATED              STATUS                        PORTS
predictive-maintenance-airflow-scheduler   infra-airflow-scheduler         "/usr/bin/dumb-init …"   airflow-scheduler   About a minute ago   Up About a minute             8080/tcp
predictive-maintenance-airflow-webserver   infra-airflow-webserver         "/usr/bin/dumb-init …"   airflow-webserver   About a minute ago   Up About a minute (healthy)   0.0.0.0:8081->8080/tcp, [::]:8081->8080/tcp
predictive-maintenance-api                 infra-api                       "uvicorn app.main:ap…"   api                 About a minute ago   Up About a minute (healthy)   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
predictive-maintenance-grafana             grafana/grafana:11.3.1          "/run.sh"                grafana             3 days ago           Up About a minute (healthy)   0.0.0.0:3000->3000/tcp, [::]:3000->3000/tc

# Проверка FastAPI health endpoint

Проверяем, что inference API доступен по HTTP и возвращает статус сервиса.

Endpoint: GET /health


Ожидаемый результат: HTTP-запрос должен вернуть JSON со статусом ok, признаком загруженной модели и deployment track.

In [19]:
%%bash

cd "$PROJECT_ROOT"
curl -s http://127.0.0.1:8000/health

{"status":"ok","model_loaded":true,"deployment_track":"single","model_alias":"champion"}

# Проверка информации о production-модели

Проверяем endpoint: GET /model/info

Он показывает, какую модель FastAPI service загрузил из MLflow Model Registry.

Ожидаемый результат:

- model_name = predictive-maintenance-model;
- model_alias = champion;
- status = loaded.

In [20]:
%%bash

cd "$PROJECT_ROOT"
curl -s http://127.0.0.1:8000/model/info

{"model_name":"predictive-maintenance-model","model_alias":"champion","model_uri":"models:/predictive-maintenance-model@champion","status":"loaded"}

# Проверка inference endpoint `/predict`

Проверяем endpoint: POST /predict

Этот endpoint принимает полный набор признаков в request body и возвращает прогноз отказа оборудования, вероятность отказа, уровень риска и рекомендуемое действие.

Ожидаемый результат: HTTP-запрос должен вернуть JSON с полями failure_probability, prediction, risk_level, recommended_action, model_name и model_alias.

In [21]:
%%bash
cd "$PROJECT_ROOT"

curl -s -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "air_temperature_k": 298.1,
    "process_temperature_k": 308.6,
    "rotational_speed_rpm": 1551,
    "torque_nm": 42.8,
    "tool_wear_min": 120,
    "machine_type_H": 0,
    "machine_type_L": 0,
    "machine_type_M": 1
  }'

{"failure_probability":0.01965044927029856,"prediction":0,"risk_level":"low","recommended_action":"continue_normal_operation","model_name":"predictive-maintenance-model","model_alias":"champion"}

# Проверка Feast Feature Store

Feast используется для управления признаками модели.

В проекте проверяются два сценария работы Feature Store:

- offline retrieval  — получение исторических признаков для обучения
- online retrieval   — получение online-признаков из Redis для inference
  
Проверка выполняется скриптом: pipelines/check_feature_store.py

Скрипт проверяет offline retrieval, материализацию признаков в Redis Online Store и online retrieval по machine_id.

In [54]:
%%bash
cd "$PROJECT_ROOT"

. .venv/bin/activate
FEAST_REDIS_CONNECTION_STRING=localhost:16379 python pipelines/check_feature_store.py

Offline feature retrieval passed.
   machine_id           event_timestamp  ...  machine_type_L  machine_type_M
0           1 2026-01-01 00:00:00+00:00  ...               0               1
1           2 2026-01-01 01:00:00+00:00  ...               1               0
2           3 2026-01-01 02:00:00+00:00  ...               1               0

[3 rows x 10 columns]
Materializing 1 feature views from 2026-01-01 00:00:00+00:00 to 2027-03-01 00:00:00+00:00 into the redis online store.

machine_sensor_features:
Feature materialization to Redis passed.
Online feature retrieval from Redis passed.
{'machine_id': [1, 2, 3], 'machine_type_L': [0, 1, 1], 'machine_type_M': [1, 0, 0], 'machine_type_H': [0, 0, 0], 'torque_nm': [42.79999923706055, 46.29999923706055, 49.400001525878906], 'rotational_speed_rpm': [1551, 1408, 1498], 'tool_wear_min': [0, 3, 5], 'air_temperature_k': [298.1000061035156, 298.20001220703125, 298.1000061035156], 'process_temperature_k': [308.6000061035156, 308.70001220703125, 3

# Проверка production-like inference через Feast Redis

Проверяем endpoint: POST /predict/from-feature-store

Этот endpoint принимает только machine_id, получает online-признаки из Feast Redis Online Store и затем выполняет inference champion-моделью из MLflow Model Registry.

Эта проверка важна, потому что она показывает не просто ручную передачу признаков в API, а прод путь:

**machine_id -> Feast Redis Online Store -> FastAPI -> MLflow champion model -> prediction**

In [22]:
%%bash
cd "$PROJECT_ROOT"

curl -s -X POST http://127.0.0.1:8000/predict/from-feature-store \
  -H "Content-Type: application/json" \
  -d '{"machine_id": 1}'

{"failure_probability":0.014700660952716337,"prediction":0,"risk_level":"low","recommended_action":"continue_normal_operation","model_name":"predictive-maintenance-model","model_alias":"champion"}

# Проверка FastAPI Prometheus metrics endpoint

FastAPI service отдаёт технические метрики на endpoint: GET /metrics

Prometheus использует этот endpoint для сбора метрик inference service.

Проверяем наличие основных метрик:

- predict_requests_total;
- predict_errors_total;
- predict_latency_seconds;
- feature_retrieval_requests_total;
- feature_retrieval_errors_total;
- feature_retrieval_latency_seconds.

In [23]:
%%bash

cd "$PROJECT_ROOT"
curl -s http://127.0.0.1:8000/metrics | grep -E "predict_requests_total|predict_errors_total|predict_latency_seconds|feature_retrieval_requests_total|feature_retrieval_errors_total|feature_retrieval_latency_seconds" | head -n 40

# HELP predict_requests_total Total number of prediction requests
# TYPE predict_requests_total counter
predict_requests_total 2.0
# HELP predict_errors_total Total number of prediction errors
# TYPE predict_errors_total counter
predict_errors_total 0.0
# HELP predict_latency_seconds Prediction latency in seconds
# TYPE predict_latency_seconds histogram
predict_latency_seconds_bucket{le="0.005"} 0.0
predict_latency_seconds_bucket{le="0.01"} 0.0
predict_latency_seconds_bucket{le="0.025"} 0.0
predict_latency_seconds_bucket{le="0.05"} 0.0
predict_latency_seconds_bucket{le="0.075"} 1.0
predict_latency_seconds_bucket{le="0.1"} 1.0
predict_latency_seconds_bucket{le="0.25"} 1.0
predict_latency_seconds_bucket{le="0.5"} 2.0
predict_latency_seconds_bucket{le="0.75"} 2.0
predict_latency_seconds_bucket{le="1.0"} 2.0
predict_latency_seconds_bucket{le="2.5"} 2.0
predict_latency_seconds_bucket{le="5.0"} 2.0
predict_latency_seconds_bucket{le="7.5"} 2.0
predict_latency_seconds_bucket{le="10.0"} 2.0
pre

# Проверка Prometheus health endpoint

Prometheus используется для сбора технических метрик FastAPI service, Node Exporter и самого Prometheus.

Проверяем health endpoint: GET /-/healthy

In [24]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:9090/-/healthy

Prometheus Server is Healthy.


# Проверка Prometheus targets

Проверяем, что Prometheus видит основные адреса сервисов:

- `predictive-maintenance-api` - метрики FastAPI: latency, ошибки, количество запросов, feature retrieval.
- `node-exporter` - метрики машины/инфраструктуры: CPU, память, диск итд
- `prometheus` — метрики самого Prometheus.

Нужно показать, что мониторинг не только запущен, но и реально собирает метрики с сервисов.

In [25]:
%%bash
cd "$PROJECT_ROOT"

curl -s "http://127.0.0.1:9090/api/v1/targets" \
  | python -m json.tool \
  | grep -E '"job"|"health"|"scrapeUrl"' \
  | head -n 40

                    "job": "node-exporter"
                    "job": "node-exporter"
                "scrapeUrl": "http://node-exporter:9100/metrics",
                "health": "up",
                    "job": "predictive-maintenance-api"
                    "job": "predictive-maintenance-api"
                "scrapeUrl": "http://api:8000/metrics",
                "health": "up",
                    "job": "prometheus"
                    "job": "prometheus"
                "scrapeUrl": "http://prometheus:9090/metrics",
                "health": "up",


# Проверка Prometheus alert rules / SLO rules

Проверяем, что Prometheus загрузил alert rules, связанные с SLO inference service.

В проекте используются правила для контроля:

- высокой latency;
- высокого error rate;
- недоступности API;
- ошибок online feature retrieval из Feast Redis.

In [26]:
%%bash
cd "$PROJECT_ROOT"

curl -s "http://127.0.0.1:9090/api/v1/rules" \
  | python -m json.tool \
  | grep -E '"name": "PredictiveMaintenance|state|health|query' \
  | head -n 80

                        "state": "inactive",
                        "name": "PredictiveMaintenanceHighLatency",
                        "query": "histogram_quantile(0.95, sum by (le) (rate(predict_latency_seconds_bucket[5m]))) > 1",
                        "health": "ok",
                        "state": "inactive",
                        "name": "PredictiveMaintenanceHighErrorRate",
                        "query": "(sum(rate(predict_errors_total[5m])) / clamp_min(sum(rate(predict_requests_total[5m])), 1)) > 0.01",
                        "health": "ok",
                        "state": "inactive",
                        "name": "PredictiveMaintenanceApiDown",
                        "query": "up{job=\"predictive-maintenance-api\"} == 0",
                        "health": "ok",
                        "state": "inactive",
                        "name": "PredictiveMaintenanceFeatureRetrievalErrors",
                        "query": "(sum(rate(feature_retrieval_errors_total[5m])) / 

# Проверка Grafana health endpoint

Grafana используется для визуализации метрик Prometheus.

Проверяем health endpoint Grafana: GET /api/health

In [27]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:3000/api/health

{
  "database": "ok",
  "version": "11.3.1",
  "commit": "64b556c137a1d9bcacd19ccb16c4cf138c78ca40"
}

# Проверка MLflow Tracking Server

MLflow используется для отслеживания экспериментов, хранения метрик обучения, артефактов моделей и ведения реестра моделей.

Проверяем, что MLflow-сервер доступен по HTTP.

В локальном Docker Compose контуре веб-интерфейс MLflow открыт на порту: http://127.0.0.1:5050

In [28]:
%%bash
cd "$PROJECT_ROOT"

curl -s -I http://127.0.0.1:5050 | head

HTTP/1.1 200 OK
Server: gunicorn
Date: Sat, 30 May 2026 18:12:18 GMT
Connection: close
Content-Disposition: inline; filename=index.html
Content-Type: text/html; charset=utf-8
Content-Length: 645
Last-Modified: Mon, 18 Nov 2024 15:25:23 GMT
Cache-Control: no-cache
ETag: "1731943523.0-645-3609271048"


# Проверка Airflow

Airflow используется для оркестрации пайплайна обучения и продвижения модели.

Проверяем служебный endpoint Airflow: GET /health

В локальном Docker Compose контуре веб-интерфейс Airflow открыт на порту: http://127.0.0.1:8081

In [29]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:8081/health

{"dag_processor": {"latest_dag_processor_heartbeat": null, "status": null}, "metadatabase": {"status": "healthy"}, "scheduler": {"latest_scheduler_heartbeat": "2026-05-30T18:13:14.209696+00:00", "status": "healthy"}, "triggerer": {"latest_triggerer_heartbeat": null, "status": null}}

# Проверка DAG в Airflow

Проверяем, что Airflow видит DAG пайплайна обучения: predictive_maintenance_training_pipeline

Этот DAG отвечает за запуск этапов подготовки данных, построения признаков, проверки Feature Store, обучения моделей и продвижения лучшей модели.

In [30]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.yml exec airflow-scheduler \
  airflow dags list | grep predictive_maintenance_training_pipeline

predictive_maintenance_training_pipeline | /opt/airflow/dags/predictive_maintenance_training_dag.py | airflow | False    


# Запуск Airflow DAG

Запускаем DAG `predictive_maintenance_training_pipeline`.

In [31]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.yml exec airflow-scheduler \
  airflow dags trigger predictive_maintenance_training_pipeline

[2026-05-30T18:16:23.601+0000] {__init__.py:43} INFO - Loaded API auth backend: airflow.api.auth.backend.session
     |                     |                     |                     |                     |          |                  | last_scheduling_deci |                     |          |            |       
conf | dag_id              | dag_run_id          | data_interval_start | data_interval_end   | end_date | external_trigger | sion                 | logical_date        | run_type | start_date | state 
=====+=====================+=====================+=====================+=====================+==========+==================+======================+=====================+==========+============+=======
{}   | predictive_maintena | manual__2026-05-30T | 2026-05-30          | 2026-05-30          | None     | True             | None                 | 2026-05-30          | manual   | None       | queued
     | nce_training_pipeli | 18:16:23+00:00      | 18:16:23+00:00      | 18:16:23+0

# Проверка статуса запущенного DAG

После запуска DAG проверяем его состояние.  
Сначала DAG может быть в статусе `queued` или `running`. После завершения успешного пайплайна ожидаемый статус `success`.

In [32]:
%%bash
cd "$PROJECT_ROOT"

sleep 20

docker compose -f infra/docker-compose.yml exec airflow-scheduler \
  airflow dags list-runs -d predictive_maintenance_training_pipeline | head -n 20

dag_id                                   | run_id                            | state   | execution_date            | start_date                       | end_date                        
=========================================+===================================+=========+===========================+==================================+=================================
predictive_maintenance_training_pipeline | manual__2026-05-30T18:16:23+00:00 | success | 2026-05-30T18:16:23+00:00 | 2026-05-30T18:16:24.756295+00:00 | 2026-05-30T18:16:46.837752+00:00
predictive_maintenance_training_pipeline | manual__2026-05-27T12:32:55+00:00 | success | 2026-05-27T12:32:55+00:00 | 2026-05-27T12:32:55.903753+00:00 | 2026-05-27T12:33:16.943774+00:00
predictive_maintenance_training_pipeline | manual__2026-05-27T12:27:47+00:00 | failed  | 2026-05-27T12:27:47+00:00 | 2026-05-27T12:27:48.531776+00:00 | 2026-05-27T12:27:57.620754+00:00
                                                                           

# Повторная проверка production-модели после запуска DAG

После успешного запуска Airflow DAG повторно проверяем endpoint `/model/info`.

Это подтвердит, что FastAPI service видит production-модель из MLflow Model Registry и загружает её по alias `champion`.

In [33]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:8000/model/info

{"model_name":"predictive-maintenance-model","model_alias":"champion","model_uri":"models:/predictive-maintenance-model@champion","status":"loaded"}

# Проверка Node Exporter

Node Exporter используется для сбора инфраструктурных метрик машины таких как CPU, памяти, диска и других системных показателей.

Проверяем, что Node Exporter отдаёт метрики на endpoint: GET /metrics

В локальном контуре Node Exporter доступен на порту 9100.

In [34]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:9100/metrics | grep -E "node_cpu_seconds_total|node_memory|node_filesystem" | head -n 20

# HELP node_cpu_seconds_total Seconds the CPUs spent in each mode.
# TYPE node_cpu_seconds_total counter
node_cpu_seconds_total{cpu="0",mode="idle"} 1479.66
node_cpu_seconds_total{cpu="0",mode="iowait"} 2.1
node_cpu_seconds_total{cpu="0",mode="irq"} 0
node_cpu_seconds_total{cpu="0",mode="nice"} 0
node_cpu_seconds_total{cpu="0",mode="softirq"} 13.5
node_cpu_seconds_total{cpu="0",mode="steal"} 0
node_cpu_seconds_total{cpu="0",mode="system"} 7.63
node_cpu_seconds_total{cpu="0",mode="user"} 30.28
node_cpu_seconds_total{cpu="1",mode="idle"} 1502.37
node_cpu_seconds_total{cpu="1",mode="iowait"} 2.42
node_cpu_seconds_total{cpu="1",mode="irq"} 0
node_cpu_seconds_total{cpu="1",mode="nice"} 0
node_cpu_seconds_total{cpu="1",mode="softirq"} 5.44
node_cpu_seconds_total{cpu="1",mode="steal"} 0
node_cpu_seconds_total{cpu="1",mode="system"} 4.72
node_cpu_seconds_total{cpu="1",mode="user"} 17.97
node_cpu_seconds_total{cpu="10",mode="idle"} 1511.03
node_cpu_seconds_total{cpu="10",mode="iowait"} 0.63


# Проверка инфраструктурных метрик через Prometheus

Теперь проверяем не только сам Node Exporter, но и то что Prometheus успешно собирает его метрики.

Для этого запрашиваем метрику `node_cpu_seconds_total` через API Prometheus.

In [35]:
%%bash
cd "$PROJECT_ROOT"
curl -s "http://127.0.0.1:9090/api/v1/query?query=node_cpu_seconds_total" \
  | python -m json.tool \
  | grep -E '"status"|"__name__"|"job"|"instance"' \
  | head -n 20

    "status": "success",
                    "__name__": "node_cpu_seconds_total",
                    "instance": "node-exporter:9100",
                    "job": "node-exporter",
                    "__name__": "node_cpu_seconds_total",
                    "instance": "node-exporter:9100",
                    "job": "node-exporter",
                    "__name__": "node_cpu_seconds_total",
                    "instance": "node-exporter:9100",
                    "job": "node-exporter",
                    "__name__": "node_cpu_seconds_total",
                    "instance": "node-exporter:9100",
                    "job": "node-exporter",
                    "__name__": "node_cpu_seconds_total",
                    "instance": "node-exporter:9100",
                    "job": "node-exporter",
                    "__name__": "node_cpu_seconds_total",
                    "instance": "node-exporter:9100",
                    "job": "node-exporter",
                    "__name__": "node_c

# Проверка data drift report через Evidently

Evidently используется для контроля дрифта входных данных.

Скрипт `pipelines/check_data_drift.py` сравнивает reference dataset и current dataset, строит HTML-отчёт и сохраняет JSON summary.

Ожидаемые артефакты:

```text
reports/evidently/data_drift_report.html
reports/evidently/data_drift_summary.json
```

In [36]:
%%bash
cd "$PROJECT_ROOT"

. .venv/bin/activate
make drift-check

python pipelines/check_data_drift.py
Evidently data drift report was created.
HTML report: reports/evidently/data_drift_report.html
JSON summary: reports/evidently/data_drift_summary.json
{
  "reference_rows": 7000,
  "current_rows": 3000,
  "features": [
    "air_temperature_k",
    "process_temperature_k",
    "rotational_speed_rpm",
    "torque_nm",
    "tool_wear_min",
    "machine_type_H",
    "machine_type_L",
    "machine_type_M"
  ],
  "dataset_drift": false,
  "share_of_drifted_columns": 0.25,
  "number_of_drifted_columns": 2,
  "html_report_path": "reports/evidently/data_drift_report.html"
}


# Проверка артефактов Evidently

Проверяем, что после запуска drift-check были созданы оба артефакта

- HTML-отчёт для визуального анализа;
- JSON summary для машинно-читаемого результата.

In [37]:
%%bash
cd "$PROJECT_ROOT"

ls -lh reports/evidently/data_drift_report.html
ls -lh reports/evidently/data_drift_summary.json

-rw-r--r--@ 1 perceivery  staff   3.1M May 30 21:21 reports/evidently/data_drift_report.html
-rw-r--r--@ 1 perceivery  staff   420B May 30 21:21 reports/evidently/data_drift_summary.json


# Запуск canary-контура

Теперь запускаем отдельный canary-контур из файла: infra/docker-compose.canary.yml

Он поднимает:

- stable inference service;
- canary inference service;
- Nginx gateway на порту 8010.

Canary gateway нужен для демонстрации постепенного переключения трафика между stable и canary API-сервисами.

In [38]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.canary.yml up -d --build

#1 [internal] load local bake definitions
#1 reading from stdin 1.05kB done
#1 DONE 0.0s

#2 [canary internal] load build definition from Dockerfile.api
#2 transferring dockerfile: 680B done
#2 DONE 0.0s

#3 [stable internal] load metadata for docker.io/library/python:3.12-slim
#3 DONE 1.9s

#4 [canary internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [canary internal] load build context
#5 transferring context: 751B done
#5 DONE 0.0s

#6 [canary 1/7] FROM docker.io/library/python:3.12-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203
#6 resolve docker.io/library/python:3.12-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203 done
#6 DONE 0.0s

#7 [canary 5/7] RUN python -m pip install --upgrade pip setuptools wheel     && python -m pip install -r /app/requirements.txt
#7 CACHED

#8 [canary 2/7] WORKDIR /app
#8 CACHED

#9 [canary 3/7] RUN apt-get update     && apt-get install -y --no-install-recommends cur

 infra-canary  Built
 infra-stable  Built
time="2026-05-30T21:23:43+03:00" level=warning msg="Found orphan containers ([predictive-maintenance-airflow-scheduler predictive-maintenance-airflow-webserver predictive-maintenance-airflow-init predictive-maintenance-api predictive-maintenance-prometheus predictive-maintenance-node-exporter predictive-maintenance-grafana predictive-maintenance-redis predictive-maintenance-mlflow predictive-maintenance-postgres molvit-minio]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."
 Container predictive-maintenance-api-canary  Recreate
 Container predictive-maintenance-api-stable  Recreate
 Container predictive-maintenance-api-stable  Recreated
 Container predictive-maintenance-api-canary  Recreated
 Container predictive-maintenance-api-canary  Starting
 Container predictive-maintenance-api-stable  Starting
 Container predictive-maintenance-api-stabl

# Проверка статусов canary-контура

Проверяем, что поднялись оба экземпляра inference service:

- `stable`;
- `canary`;

а также Nginx gateway, который принимает внешний трафик на порту `8010`.

In [39]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.canary.yml ps

NAME                                       IMAGE                           COMMAND                  SERVICE             CREATED          STATUS                             PORTS
predictive-maintenance-airflow-scheduler   infra-airflow-scheduler         "/usr/bin/dumb-init …"   airflow-scheduler   30 minutes ago   Up 29 minutes                      8080/tcp
predictive-maintenance-airflow-webserver   infra-airflow-webserver         "/usr/bin/dumb-init …"   airflow-webserver   30 minutes ago   Up 29 minutes (healthy)            0.0.0.0:8081->8080/tcp, [::]:8081->8080/tcp
predictive-maintenance-api                 infra-api                       "uvicorn app.main:ap…"   api                 30 minutes ago   Up 30 minutes (healthy)            0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
predictive-maintenance-api-canary          infra-canary                    "uvicorn app.main:ap…"   canary              48 seconds ago   Up 47 seconds (healthy)            8000/tcp
predictive-maintenance-api-s

# Проверка canary gateway

Проверяем, что Nginx доступен на порту `8010` и проксирует запросы к inference service.

Endpoint: GET /health

Ожидаемый результат: JSON с status = ok и полем deployment_track, которое показывает какой backend ответил на запрос (stable или canary).

In [40]:
%%bash
cd "$PROJECT_ROOT"

curl -s http://127.0.0.1:8010/health

{"status":"ok","model_loaded":true,"deployment_track":"stable","model_alias":"champion"}

# Проверка распределения трафика через canary

Проверяем, как Nginx-шлюз распределяет запросы между двумя версиями сервиса:

- `stable` — стабильная версия сервиса;
- `canary` — тестовая версия сервиса.

Скрипт `scripts/check_canary_distribution.sh` выполняет несколько запросов к endpoint `/health` через Nginx-шлюз и считает, какая версия сервиса ответила на запрос.

In [41]:
%%bash
cd "$PROJECT_ROOT"

N=50 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

  50 {"status":"ok","model_loaded":true,"deployment_track":"stable","model_alias":"champion"}


# Переключение трафика в режим 50/50

Проверяем механизм canary переключения.

Скрипт `scripts/switch_canary_50_50.sh` меняет конфигурацию Nginx-шлюза так, чтобы примерно половина запросов шла в `stable`, а половина в `canary`.

После переключения повторно проверим распределение запросов.

In [42]:
%%bash
cd "$PROJECT_ROOT"

scripts/switch_canary_50_50.sh
sleep 5
N=50 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

 Container predictive-maintenance-canary-gateway  Restarting
 Container predictive-maintenance-canary-gateway  Started


Traffic switched to canary mode 50/50.
  25 {"status":"ok","model_loaded":true,"deployment_track":"canary","model_alias":"champion"}
  25 {"status":"ok","model_loaded":true,"deployment_track":"stable","model_alias":"champion"}


# Переключение 100% трафика на canary

Проверяем сценарий полного переключения трафика на canary версию сервиса.

Скрипт `scripts/switch_canary_100.sh` меняет конфигурацию Nginx-шлюза так, чтобы все запросы через порт `8010` направлялись в `canary`.

In [43]:
%%bash
cd "$PROJECT_ROOT"

scripts/switch_canary_100.sh
sleep 5
N=20 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

 Container predictive-maintenance-canary-gateway  Restarting
 Container predictive-maintenance-canary-gateway  Started


Traffic switched to 100% canary.
  20 {"status":"ok","model_loaded":true,"deployment_track":"canary","model_alias":"champion"}


# Rollback возврат трафика на stable

Проверяем сценарий отката.

Скрипт `scripts/rollback_canary_to_stable.sh` возвращает конфигурацию Nginx-шлюза в безопасный режим то есть все запросы снова направляются в `stable`.

In [44]:
%%bash
cd "$PROJECT_ROOT"

scripts/rollback_canary_to_stable.sh
sleep 5
N=20 scripts/check_canary_distribution.sh | grep deployment_track | sort | uniq -c

 Container predictive-maintenance-canary-gateway  Restarting
 Container predictive-maintenance-canary-gateway  Started


Rollback completed: traffic switched back to 100% stable.
  20 {"status":"ok","model_loaded":true,"deployment_track":"stable","model_alias":"champion"}


# Итоговая проверка canary-контура после rollback

После проверки переключения трафика повторно проверяем состояние canary-контейнеров.

Важно чтобы после rollback оба сервиса `stable` и `canary` оставались работоспособными, а Nginx-шлюз продолжал отвечать на запросы.

In [45]:
%%bash
cd "$PROJECT_ROOT"

docker compose -f infra/docker-compose.canary.yml ps
curl -s http://127.0.0.1:8010/health

NAME                                       IMAGE                           COMMAND                  SERVICE             CREATED          STATUS                             PORTS
predictive-maintenance-airflow-scheduler   infra-airflow-scheduler         "/usr/bin/dumb-init …"   airflow-scheduler   36 minutes ago   Up 36 minutes                      8080/tcp
predictive-maintenance-airflow-webserver   infra-airflow-webserver         "/usr/bin/dumb-init …"   airflow-webserver   36 minutes ago   Up 36 minutes (healthy)            0.0.0.0:8081->8080/tcp, [::]:8081->8080/tcp
predictive-maintenance-api                 infra-api                       "uvicorn app.main:ap…"   api                 36 minutes ago   Up 36 minutes (healthy)            0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
predictive-maintenance-api-canary          infra-canary                    "uvicorn app.main:ap…"   canary              7 minutes ago    Up 7 minutes (healthy)             8000/tcp
predictive-maintenance-api-s

# Проверка CI/CD через GitHub Actions

GitHub Actions используется как CI-контур проекта.

В CI автоматически проверяется, что проект устанавливается и проходит тесты. Это дополняет локальную проверку `make test` то есть код проверяется не только на машине разработчика, но и в удалённом CI-окружении GitHub.

In [52]:
%%bash
cd "$PROJECT_ROOT"

ls -la .github/workflows
cat .github/workflows/ci.yml

total 8
drwxr-xr-x@ 3 perceivery  staff   96 May 27 14:45 .
drwxr-xr-x@ 3 perceivery  staff   96 May 27 14:45 ..
-rw-r--r--@ 1 perceivery  staff  585 May 27 14:29 ci.yml
name: CI

on:
  push:
    branches:
      - main
  pull_request:
    branches:
      - main

jobs:
  tests:
    name: Run tests
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip setuptools wheel
          python -m pip install -r requirements.txt

      - name: Run tests
        run: |
          python -m pytest -q


CI workflow находится в файле `.github/workflows/ci.yml`.

Он запускается при `push` и `pull_request` в ветку `main`. В workflow выполняются установка Python 3.12, установка зависимостей из `requirements.txt` и запуск тестов через `pytest`.

<h3>СI</h3>
<img src="../screenshots/10_github_actions_success.png" alt="MLflow experiments" width="900">

# Проверка Ansible Infrastructure as Code

Ansible используется как Infrastructure as Code слой для развёртывания проекта на виртуальной машине.

Docker Compose описывает состав контейнеров, а Ansible описывает воспроизводимый процесс подготовки сервера и запуска проекта:

```text
пустая VM
-> установка системных пакетов
-> установка и запуск Docker
-> копирование проекта на VM
-> сборка Docker-образов
-> запуск MLOps-инфраструктуры
-> обучение и promotion модели
-> проверка health endpoints

Основные файлы Ansible-контура:
- ansible/inventory.ini
- ansible/group_vars/all.yml
- ansible/playbook.yml
- ansible/README.md
```

In [53]:
%%bash
cd "$PROJECT_ROOT"

ls -la ansible/
echo "----- group_vars -----"
ls -la ansible/group_vars/

python - <<'PY'
import yaml

for file in [
    "ansible/group_vars/all.yml",
    "ansible/playbook.yml",
]:
    with open(file, "r", encoding="utf-8") as f:
        yaml.safe_load(f)
    print(f"YAML OK: {file}")
PY

total 32
drwxr-xr-x@  6 perceivery  staff   192 May 27 18:38 .
drwxr-xr-x@ 31 perceivery  staff   992 May 31 00:00 ..
-rw-------@  1 perceivery  staff  2522 May 27 16:17 README.md
drwxr-xr-x@  3 perceivery  staff    96 May 27 18:37 group_vars
-rw-r--r--@  1 perceivery  staff   119 May 27 16:10 inventory.ini
-rw-r--r--@  1 perceivery  staff  6066 May 27 18:09 playbook.yml
----- group_vars -----
total 8
drwxr-xr-x@ 3 perceivery  staff   96 May 27 18:37 .
drwxr-xr-x@ 6 perceivery  staff  192 May 27 18:38 ..
-rw-r--r--@ 1 perceivery  staff  701 May 27 18:11 all.yml
YAML OK: ansible/group_vars/all.yml
YAML OK: ansible/playbook.yml


Фактический запуск на виртуальной машине выполняется командой:

```bash
ansible-playbook -i ansible/inventory.ini ansible/playbook.yml
```

# Развернем на ВМ через Ansible

проверяем подключение 

In [59]:
%%bash
cd "$PROJECT_ROOT"

ansible -i ansible/inventory.ini mlops -m ping

mlops-vm | SUCCESS => {
    "changed": false,
    "ping": "pong"
}


Ansible успешно подключился к виртуальной машине `mlops-vm`.

Ответ `ping: pong` подтверждает, что SSH-доступ по ключу работает, inventory настроен корректно, и VM готова к запуску Ansible playbook для развёртывания MLOps-инфраструктуры.

Теперь проверим корректность playbook

In [60]:
%%bash
cd "$PROJECT_ROOT"

ansible-playbook -i ansible/inventory.ini ansible/playbook.yml --syntax-check


playbook: ansible/playbook.yml


Ansible playbook прошёл синтаксическую проверку.

Команда `ansible-playbook --syntax-check` подтвердила, что файл `ansible/playbook.yml` корректно читается Ansible и может быть запущен для развёртывания проекта на VM.

Запускаем проект на ВМ

In [68]:
%%bash
cd "$PROJECT_ROOT"

ansible-playbook -i ansible/inventory.ini ansible/playbook.yml


PLAY [Deploy predictive maintenance MLOps infrastructure] **********************

TASK [Gathering Facts] *********************************************************
ok: [mlops-vm]

TASK [Install base system packages] ********************************************


[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: INJECT_FACTS_AS_VARS default to `True` is deprecated, top-level facts will not be auto injected after the change. This feature will be removed from ansible-core version 2.24.
Origin: /Users/perceivery/Desktop/predictive-maintenance-mlops/ansible/playbook.yml:13:13

11         state: present
12         update_cache: true
13       when: ansible_os_family == "Debian"
               ^ column 13

Use `ansible_facts["fact_name"]` (no `ansible_` prefix) instead.



ok: [mlops-vm]

TASK [Install Docker using official convenience script if docker is absent] ****
ok: [mlops-vm]

TASK [Ensure Docker service is enabled and started] ****************************
ok: [mlops-vm]

TASK [Add deployment user to docker group] *************************************
ok: [mlops-vm]

TASK [Create remote project directory] *****************************************
ok: [mlops-vm]

TASK [Synchronize project files to VM] *****************************************
changed: [mlops-vm]

TASK [Build main Docker images] ************************************************
changed: [mlops-vm]

TASK [Build canary Docker images] **********************************************
changed: [mlops-vm]

TASK [Start core infrastructure] ***********************************************
changed: [mlops-vm]

TASK [Wait for MLflow to become available] *************************************
ok: [mlops-vm]

TASK [Run training pipeline and promote champion model] ************************
changed: [

Ansible playbook успешно развернул MLOps-инфраструктуру на виртуальной машине.

В конце выполнения получен статус:

```text
failed=0
unreachable=0
```
Это означает, что развёртывание завершилось без ошибок. На VM подняты основные сервисы проекта: FastAPI, MLflow, Airflow, Prometheus, Grafana, Node Exporter, Redis, PostgreSQL, а также canary gateway.

Также выполнен smoke-test production-like inference через Feast Redis. Ответ API содержит модель predictive-maintenance-model с alias champion, вероятность отказа, класс прогноза, уровень риска и рекомендуемое действие. 

Cервис работает на VM и использует production-like путь получения признаков и инференса.


Проверим публичные ссылки не по SSH

In [69]:
%%bash
curl -s http://158.160.13.227:8000/health

{"status":"ok","model_loaded":true,"deployment_track":"single","model_alias":"champion"}

In [70]:
%%bash
curl -I http://158.160.13.227:8000/docs

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0   960    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


HTTP/1.1 200 OK
date: Sun, 31 May 2026 09:33:50 GMT
server: uvicorn
content-length: 960
content-type: text/html; charset=utf-8



Осуществим прод инференс снаружи VM

In [71]:
%%bash
curl -s -X POST http://158.160.13.227:8000/predict/from-feature-store \
  -H "Content-Type: application/json" \
  -d '{"machine_id": 1}'

{"failure_probability":0.014700660952716337,"prediction":0,"risk_level":"low","recommended_action":"continue_normal_operation","model_name":"predictive-maintenance-model","model_alias":"champion"}

Публичные UI-ссылки 

- http://158.160.13.227:8000/docs
- http://158.160.13.227:5050
- http://158.160.13.227:8081
- http://158.160.13.227:9090
- http://158.160.13.227:3000

In [74]:
%%bash
cd "$PROJECT_ROOT"

ansible -i ansible/inventory.ini mlops -m shell -a '
cd /opt/predictive-maintenance-mlops

echo "=== docker ps ==="
docker compose -f infra/docker-compose.yml ps

echo
echo "=== local health checks inside VM ==="
curl -s -o /dev/null -w "fastapi: %{http_code}\n" http://127.0.0.1:8000/health
curl -s -o /dev/null -w "airflow: %{http_code}\n" http://127.0.0.1:8081/health
curl -s -o /dev/null -w "mlflow: %{http_code}\n" http://127.0.0.1:5050
curl -s -o /dev/null -w "prometheus: %{http_code}\n" http://127.0.0.1:9090/-/healthy
curl -s -o /dev/null -w "grafana: %{http_code}\n" http://127.0.0.1:3000/api/health
'

[ERROR]: Task failed: Failed to connect to the host via ssh: Connection timed out during banner exchange
Connection to 158.160.13.227 port 22 timed out
Origin: <adhoc 'shell' task>

{'action': 'shell', 'args': {'_raw_params': '\ncd /opt/predictive-maintenance-mlops\n\necho "=== docker ps [...]

mlops-vm | UNREACHABLE! => {
    "changed": false,
    "msg": "Task failed: Failed to connect to the host via ssh: Connection timed out during banner exchange\r\nConnection to 158.160.13.227 port 22 timed out",
    "unreachable": true
}


CalledProcessError: Command 'b'cd "$PROJECT_ROOT"\n\nansible -i ansible/inventory.ini mlops -m shell -a \'\ncd /opt/predictive-maintenance-mlops\n\necho "=== docker ps ==="\ndocker compose -f infra/docker-compose.yml ps\n\necho\necho "=== local health checks inside VM ==="\ncurl -s -o /dev/null -w "fastapi: %{http_code}\\n" http://127.0.0.1:8000/health\ncurl -s -o /dev/null -w "airflow: %{http_code}\\n" http://127.0.0.1:8081/health\ncurl -s -o /dev/null -w "mlflow: %{http_code}\\n" http://127.0.0.1:5050\ncurl -s -o /dev/null -w "prometheus: %{http_code}\\n" http://127.0.0.1:9090/-/healthy\ncurl -s -o /dev/null -w "grafana: %{http_code}\\n" http://127.0.0.1:3000/api/health\n\'\n'' returned non-zero exit status 4.

# Дадим нагрузку на инференс

In [57]:
%%bash
cd "$PROJECT_ROOT"

echo "Starting inference load for 120 seconds..."
echo "Endpoint: /predict/from-feature-store"

end=$((SECONDS + 120))

while [ $SECONDS -lt $end ]; do
  for i in $(seq 1 20); do
    curl -s -X POST http://127.0.0.1:8000/predict/from-feature-store \
      -H "Content-Type: application/json" \
      -d '{"machine_id": 1}' >/dev/null &
  done

  wait
  sleep 1
done

echo "Inference load finished."

curl -s http://127.0.0.1:8000/metrics \
  | grep -E "^predict_requests_total|^predict_errors_total|^feature_retrieval_requests_total|^feature_retrieval_errors_total"

Starting inference load for 120 seconds...
Endpoint: /predict/from-feature-store
Inference load finished.
predict_requests_total 3472.0
predict_errors_total 0.0
feature_retrieval_requests_total 3471.0
feature_retrieval_errors_total 0.0


<h3>Grafana dashboard</h3>
<img src="../screenshots/11_grafana_dashboard_after_load.png" alt="Grafana dashboard" width="900">

# Веб-интерфейсы для ручной проверки

После запуска сервисов доступны следующие веб-интерфейсы:

| Компонент | Адрес | Что проверить |
|---|---|---|
| FastAPI | `http://127.0.0.1:8000/docs` | Наличие endpoint-ов `/health`, `/model/info`, `/predict`, `/predict/from-feature-store`, `/metrics` |
| MLflow | `http://127.0.0.1:5050` | Эксперименты, запуски обучения, зарегистрированная модель `predictive-maintenance-model` |
| Airflow | `http://127.0.0.1:8081` | DAG `predictive_maintenance_training_pipeline` и последний запуск со статусом `success` |
| Prometheus targets | `http://127.0.0.1:9090/targets` | Targets `predictive-maintenance-api`, `node-exporter`, `prometheus` в состоянии `UP` |
| Prometheus rules | `http://127.0.0.1:9090/rules` | Правила `PredictiveMaintenanceHighLatency`, `PredictiveMaintenanceHighErrorRate`, `PredictiveMaintenanceApiDown`, `PredictiveMaintenanceFeatureRetrievalErrors` |
| Grafana | `http://127.0.0.1:3000` | Dashboard `Predictive Maintenance API` |
| Evidently report | `reports/evidently/data_drift_report.html` | HTML-отчёт по drift данных |
| Canary gateway | `http://127.0.0.1:8010/health` | Ответ от `stable` после rollback |

Логин и пароль для Airflow, Grafana:

```text
admin / admin
```

<h2>Скриншоты работающей системы</h2>

<h3>MLflow: эксперименты</h3>
<img src="../screenshots/01_mlflow_experiments.png" alt="MLflow experiments" width="900">

<h3>MLflow Model Registry: champion-модель</h3>
<img src="../screenshots/02_mlflow_model_registry_champion.png" alt="MLflow champion model" width="900">

<h3>Airflow: успешный запуск DAG</h3>
<img src="../screenshots/03_airflow_dag_success.png" alt="Airflow DAG success" width="900">

<h3>Prometheus targets</h3>
<img src="../screenshots/04_prometheus_targets.png" alt="Prometheus targets" width="900">

<h3>Prometheus rules</h3>
<img src="../screenshots/05_prometheus_rules1.png" alt="Prometheus rules" width="900">
<img src="../screenshots/05_prometheus_rules2.png" alt="Prometheus rules" width="900">

<h3>Grafana dashboard</h3>
<img src="../screenshots/06_grafana_dashboard1.png" alt="Grafana dashboard" width="900">
<img src="../screenshots/06_grafana_dashboard2.png" alt="Grafana dashboard" width="900">

<h3>FastAPI Swagger UI</h3>
<img src="../screenshots/07_fastapi_docs.png" alt="FastAPI docs" width="900">

<h3>Evidently drift report</h3>
<img src="../screenshots/08_evidently_report1.png" alt="Evidently drift report" width="900">
<img src="../screenshots/08_evidently_report2.png" alt="Evidently drift report" width="900">

<h3>Canary traffic switching</h3>
<img src="../screenshots/09_canary.png" alt="Canary traffic distribution" width="900">
